# Assignment 2: Text Classification with BERT

**Description:** This assignment notebook builds on the material from the
[lesson 4 notebook](https://github.com/datasci-w266/2025-summer-main/blob/master/materials/lesson_notebooks/lesson_4_BERT.ipynb), in which we fine-tuned a BERT model for the IMDB movie reviews sentiment classification task. In that notebook, we used the bert-base-cased model and applied traditional fine-tuning, with a brief class exercise at the end to try unfreezing different numbers of layers. In this assignment, we'll start with that exercise, and ask you to explore unfreezing more specific layers yourself. Then you'll search for and try different pre-trained BERT-style models.

This notebook should be run on a Google Colab leveraging a GPU. By default, when you open the notebook in Colab it will try to use a GPU. Please note that you the GPU is reuqired for Section 3 but not for Sections 1 and 2.
Since colab is providing free access to a GPU they place constraints on that access.  Therefore you might want to turn off the GPU access (Edit -> Notebook Settings) until you get to section 3.  Total runtime of the entire notebook (with solutions and a Colab GPU) should be about 1h with the majority of that time being in Section 3. If Colab tells you that you have reached your GPU limit, wait up to 24 hours and you should be able to access a GPU again.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasci-w266/2025-summer-main/blob/master/assignment/a2/Text_classification_BERT.ipynb)

The overall assignment structure is as follows:


0. Setup
  
  0.1 Libraries and Helper Functions

  0.2 Data Acquisition

  0.3. Data Preparation


1. Classification with BERT

  1.1. BERT Basics

  1.2 CLS-Token-based Classification

  1.3 Averaging of BERT Outputs

  1.4. Adding a CNN on top of BERT



**INSTRUCTIONS:**:

* Questions are always indicated as **QUESTION**, so you can search for this string to make sure you answered all of the questions. You are expected to fill out, run, and submit this notebook, as well as to answer the questions in the **answers** file as you did in a1.  Please do **not** remove the output from your notebooks when you submit them as we'll look at the output as well as your code for grading purposes.  We cannot award points if the output cells are empty.

* **### YOUR CODE HERE** indicates that you are supposed to write code.

* If you want to, you can run all of the cells in section 0 in bulk. This is setup work and no questions are in there. At the end of section 0 we will state all of the relevant variables that were defined and created in section 1.

* Finally, unless otherwise indicated your validation accuracy will be 0.65 or higher if you have correctly implemented the model.



## 0. Setup

### 0.1. Libraries and Helper Functions

This notebook requires the Hugging Face datasets and other prerequisites that you must download.  

In [1]:
!pip install -q transformers
!pip install -q torchinfo
!pip install -U -q datasets fsspec huggingface_hub # Hugging Face's dataset library
!pip install -q evaluate

Now we are ready to do the imports.

In [2]:
#@title Imports

import numpy as np

import transformers
import evaluate

from datasets import load_dataset
from torchinfo import summary

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

### 0.2 Data Acquisition


We will use the IMDB dataset delivered as part of the TensorFlow-datasets library, and split into training and test sets. For expedience, we will limit ourselves in terms of train and test examples.

In [3]:
imdb_dataset = load_dataset("imdb")

imdb_train_dataset = imdb_dataset['train'].shuffle()
imdb_dev_dataset = imdb_dataset['test'].shuffle().select(range(5000))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

It is always highly recommended to look at the data. What do the records look like? Are they clean or do they contain a lot of cruft (potential noise)?

In [4]:
imdb_train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [5]:
for i in range(4):
  print(imdb_train_dataset['text'][i])
  print(imdb_train_dataset['label'][i])
  print()

Oh dear, Oh dear. I started watching this not knowing what to expect. I couldn't believe what I was seeing. There were times when I thought it was a comedy. I loved how the government's plan to capture the terrorist leader is to air drop in one man, who is unarmed, and expect him to capture him and escape with a rocket pack. If only it were really that easy. I've finally found a movie worse than "Plan 9 From Outer Space".
0

This woman never stops talking throughout the movie. She memorized every line, and delivered all without a bit of natural emotion. She also has a most uncharming lisp, and the pitch of her voice sounds like nails on a blackboard. This film has WAY too much Betsy Drake, and not enough Cary Grant, who carried what little was left of the film entirely on his own.
0

This is one of my favourite kung-fu films and is regarded as one the most popular Shaw Brothers from the late 70's. The plot is interesting and twisty, the characters are cool each with their own style - t

In [6]:
imdb_train_dataset.features['label'].names

['neg', 'pos']

For convenience, in this assignment we will define a sequence length and truncate all records at that length. For records that are shorter than our defined sequence length we will add padding characters to insure that our input shapes are consistent across all records.

In [7]:
MAX_SEQUENCE_LENGTH = 100

## 0.3. Data Preparation

We will need to tokenize the text into vocab_ids to pass into a BERT model. To do so, we'll need to use the specific tokenizer that goes with the model we're using. In this notebook, we will try several different BERT-style models. Let's
first write a function that will take the text from our dataset and a tokenizer, and encode the text using that tokenizer. Then we'll apply the function to our dataset for each tokenizer and model.

In [8]:
def preprocess_imdb(data, tokenizer):
    review_text = data['text']

    encoded = tokenizer.batch_encode_plus(
            review_text,
            max_length=MAX_SEQUENCE_LENGTH,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="pt"
        )

    return encoded


## 1. BERT-based Classification Models

Now we turn to classification with BERT. We will perform classifications with various models that are based on pre-trained BERT models.  If you turn off GPU access while coding and debugging the setup steps, make sure you change the Notebook settings so you can access a GPU when you're ready to train the models.


### 1.1. Basics

Let us first explore some basics of BERT. We'll start by loading the first pretrained BERT model and tokenizer that we'll use ('bert-base-cased').

To explore just the pre-trained portion of the model, we'll use the AutoModel class (equivalent to BertModel, but works for any architecture including BERT). This class gives us the pre-trained model layers up until the last hidden layer (but not any output layer).

My own interpretation: In other words, this section is setting us up to use a pretrained BERT model (i.e., bert-base-cased) as a text encoder, which means: converting raw text into rich, contextual embeddings we can later use for classification.

Importantly, we're not using the classification head yet, and we're just loading the base BERT model (everything up to the final hidden layer).

In [9]:
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
bert_model = AutoModel.from_pretrained('bert-base-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

The AutoTokenizer loads a tokenizer that’s compatible with the 'bert-base-cased' model, which is important because BERT expects input in a specific format:
- WordPiece tokenization
- Special tokens like [CLS], [SEP]
- Padding, truncation, attention masks

The AutoModel, on the other hand, loads the pretrained BERT model, but only the encoder part (so again, no classification head).

More specifically, it loads 12 Transformer layers from BERT (because bert-base has 12 layers), and it outputs the last hidden states (vectors) for each input token.
- Shape of the output is normally (batch_size, sequence_length, hidden_size), where the hidden_size is 768 for the bert-base-cased

Also, we use the AutoModel and not the BertModel directly because the AutoModel is more flexible:
- We can swap in other architectures (like RoBERTa or DistilBERT) just by changing the string 'bert-base-cased'.
  - It’s part of the Hugging Face Transformers library design philosophy of using unified interfaces.

Lastly, this is what bert-base entails: 12-layer, 768-hidden, 12-heads, 110M parameters
- cased: It distinguishes between uppercase and lowercase words.
  - So "Apple" and "apple" get different embeddings, which is useful for proper nouns, ticker symbols, names, etc.

Let's look at a couple of example sentences:

In [10]:
test_input = ['this bank is closed on Sunday', 'the steepest bank of the river is dangerous']

Apply the BERT tokenizer to tokenize them:

In [11]:
tokenized_input = bert_tokenizer(test_input,
                                 max_length=12,
                                 truncation=True,
                                 padding='max_length',
                                 return_tensors='pt')  # pt means pytorch tensors

tokenized_input

{'input_ids': tensor([[ 101, 1142, 3085, 1110, 1804, 1113, 3625,  102,    0,    0,    0,    0],
        [ 101, 1103, 9458, 2556, 3085, 1104, 1103, 2186, 1110, 4249,  102,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]])}

input_ids - These are the token IDs, or numerical encodings of tokens using the WordPiece vocabulary of bert-base-cased.
- 101 is the special [CLS] token (marks the start)
- 102 is the special [SEP] token (marks the end)
- 0 is used for padding (because of padding='max_length')
- **Important**: Btw, "steepest" is separated into "steep" and "###est" by BERT due to **WordPiece tokenization**, which splits unknown or complex words into subword units; this is why we see token offsets that don’t always line up with word positions

token_type_ids - These tell BERT which segment a token belongs to:
- Used in tasks like question answering where input = [CLS] question [SEP] context [SEP]
- In single-sentence classification (like this one), everything gets 0

attention_mask - This tells BERT which tokens to pay attention to:
- 1 = real token
- 0 = padding token (don’t attend to this during self-attention)

Lastly, the output dictionary is a torch.Tensor, which can be fed directly into a PyTorch-based BERT model.

 **QUESTION:**

 1.a  Why do the attention_masks have 4 and 1 zeros, respectively?  Choose the correct one and enter it in the answers file.

  *  For the first example the last four tokens belong to a different segment. For the second one it is only the last token.

  *  **For the first example 4 positions are padded while for the second one it is only one.**

In [12]:
### YOUR CODE HERE

bert_output = bert_model(**tokenized_input)
bert_output


### END YOUR CODE

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.3945,  0.0420,  0.0648,  ...,  0.0505,  0.2236,  0.2424],
         [-0.0946,  0.0667, -0.0361,  ...,  0.2193, -0.0697,  0.7445],
         [ 0.0056,  0.3132, -0.1798,  ...,  0.1956, -0.1061,  0.4777],
         ...,
         [ 0.2227, -0.1156,  0.1585,  ...,  0.3003,  0.0163,  0.5133],
         [ 0.3164, -0.1099,  0.2366,  ...,  0.1092, -0.1434,  0.3284],
         [ 0.3483, -0.1008,  0.2690,  ...,  0.1271, -0.1843,  0.2618]],

        [[ 0.4451,  0.2227, -0.0997,  ..., -0.2374,  0.1272,  0.0778],
         [ 0.0741, -0.3181, -0.1192,  ..., -0.0668, -0.3062,  0.4692],
         [ 0.3146,  0.6266,  0.0061,  ..., -0.0370, -0.0846,  0.7268],
         ...,
         [ 0.6999, -0.1163,  0.0161,  ..., -0.4744,  0.0573,  0.2183],
         [ 0.5603,  0.0854, -0.9192,  ..., -0.3102, -0.0938,  0.3491],
         [-0.2686,  0.1133,  0.0756,  ...,  0.3738,  0.0074,  0.1668]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_ou

Basically, the bert_model(**tokenized_input) step passes the tokenized input into the BERT model.

The **tokenized_input syntax unpacks the dictionary (so BERT receives input_ids, attention_mask, etc.), which gives us access to the hidden states or final token representations.

 **QUESTION:**

 1.b How many outputs are there?

 Enter your code below.

In [13]:
### YOUR CODE HERE

#b. -> print it out
print("Printing out the BERT output:")
print(bert_output)
print("\nShape of first BERT output:", bert_output[0].shape)
print("Shape of second BERT output:", bert_output[1].shape)
print("\nOverall, there appears to be 2 outputs.")


### END YOUR CODE

Printing out the BERT output:
BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.3945,  0.0420,  0.0648,  ...,  0.0505,  0.2236,  0.2424],
         [-0.0946,  0.0667, -0.0361,  ...,  0.2193, -0.0697,  0.7445],
         [ 0.0056,  0.3132, -0.1798,  ...,  0.1956, -0.1061,  0.4777],
         ...,
         [ 0.2227, -0.1156,  0.1585,  ...,  0.3003,  0.0163,  0.5133],
         [ 0.3164, -0.1099,  0.2366,  ...,  0.1092, -0.1434,  0.3284],
         [ 0.3483, -0.1008,  0.2690,  ...,  0.1271, -0.1843,  0.2618]],

        [[ 0.4451,  0.2227, -0.0997,  ..., -0.2374,  0.1272,  0.0778],
         [ 0.0741, -0.3181, -0.1192,  ..., -0.0668, -0.3062,  0.4692],
         [ 0.3146,  0.6266,  0.0061,  ..., -0.0370, -0.0846,  0.7268],
         ...,
         [ 0.6999, -0.1163,  0.0161,  ..., -0.4744,  0.0573,  0.2183],
         [ 0.5603,  0.0854, -0.9192,  ..., -0.3102, -0.0938,  0.3491],
         [-0.2686,  0.1133,  0.0756,  ...,  0.3738,  0.0074,  0.1668]]],
       grad_fn=<NativeL

My own understanding of things:

1. last_hidden_state is the first bert_output with shape (batch_size, sequence_length, hidden_size)
- In our case: (2, 12, 768)
  - 2 sentences
  - 12 tokens per sentence (b/c max_length=12)
  - 768 is the dimensionality of each token embedding in bert-base-cased
- This output contains the contextual embedding for each token in the input, after going through all 12 transformer layers
  - E.g., bert_output[0][0][3] is the vector for the 4th token of the first sentence
- This is useful if we want to extract per-token embeddings, or if we want to use the [CLS] token's embedding: bert_output[0][:, 0, :] (often used as a "sentence-level" embedding)

2. pooler_output is the second bert_output with shape (batch_size, hidden_size)
- In our case: (2, 768), where each row corresponds to a sentence
- This contains a *fixed-size embedding for the entire sentence*
  - Btw, it's derived from the [CLS] token (position 0), passed through a fully connected layer + tanh activation, and it's more "task-tuned" than the raw [CLS] embedding
- This is useful if we're doing sentence-level classification (e.g., sentiment or topic classification)
  - In particular, this is often the input to a final dense-to-softmax layer in sequence classification


**QUESTION:**

1.c Which output do we need to use to get token-level embeddings?

**the first**

the second

Put your answer in the answers file.



**QUESTION:**

 1.d In the tokenized input, which input_id number (i.e. the vocabulary id) corresponds to 'bank' in the two sentences? ('bert_tokenizer.tokenize()' may come in handy.. and don't forget the CLS token! )


**QUESTION:**

 1.e In the array of tokens, which position index number corresponds to 'bank' in the first sentence? ('bert_tokenizer.tokenize()' may come in handy.. and don't forget the CLS token! )

In [14]:
### YOUR CODE HERE

#d/e. -> Look at tokens generated by the bert tokenizer for the first example

# Question 1.d Getting the input_id number that corresponds to "bank"
print("Question 1.d: In the first sentence:", bert_tokenizer.tokenize(test_input[0]))
print("After including the [CLS] token, 'bank' appears to be in index 2.\n")
print("In the second sentence:", bert_tokenizer.tokenize(test_input[1]))
print("After including the [CLS] token, 'bank' appears to be in index 4.\n")
print("Getting the vocabulary id from the tokenized_input, we can get the following:")
print(" - For index 2 in sentence 1:", tokenized_input['input_ids'][0][2].numpy())
print(" - For index 4 in sentence 2:", tokenized_input['input_ids'][1][4].numpy())
print("And we see that the input_id numbers match, and they end up being:",
      tokenized_input['input_ids'][0][2].numpy(), "\n")

# Question 1.e Getting the position index number corresponding to "bank" in the first sentence
print("Question 1.e: As we saw in the previous code, 'bank' appears to at index 2 for sentence 1.")


### END YOUR CODE

Question 1.d: In the first sentence: ['this', 'bank', 'is', 'closed', 'on', 'Sunday']
After including the [CLS] token, 'bank' appears to be in index 2.

In the second sentence: ['the', 'steep', '##est', 'bank', 'of', 'the', 'river', 'is', 'dangerous']
After including the [CLS] token, 'bank' appears to be in index 4.

Getting the vocabulary id from the tokenized_input, we can get the following:
 - For index 2 in sentence 1: 3085
 - For index 4 in sentence 2: 3085
And we see that the input_id numbers match, and they end up being: 3085 

Question 1.e: As we saw in the previous code, 'bank' appears to at index 2 for sentence 1.


**QUESTION:**

1.f Which array position index number corresponds to 'bank' in the second sentence?

In [15]:
### YOUR CODE HERE

#f. -> Look at tokenization for the second example
# Question 1.f Getting the position index number corresponding to "bank" in the second sentence
print("Question 1.f: As we saw in the previous code, 'bank' appears to at index 4 for sentence 2.")

### END YOUR CODE

Question 1.f: As we saw in the previous code, 'bank' appears to at index 4 for sentence 2.


**QUESTION:**

 1.g What is the cosine similarity between the BERT embeddings for the two occurences of 'bank' in the two sentences?

In [16]:
### YOUR CODE HERE

#g.  -> get the vectors and calculate cosine similarity between the two 'bank' BERT embeddings

# Defining a helpful cosine_similarities function from the lecture notebook:
def cosine_similarities(vecs):
    for v_1 in vecs:
        similarities = ''
        for v_2 in vecs:
            similarities += ('\t' + str(np.dot(v_1, v_2)/np.sqrt(np.dot(v_1, v_1) * np.dot(v_2, v_2)))[:4])
        print(similarities)

# Getting the cosine similarities between the BERT embeddings for "bank" in the two sentences
bank_1 = bert_output[0][0, 2]
bank_2 = bert_output[0][1, 4]

banks = [
    bank_1.detach().numpy(),  # .detach() just gets this tensor away from the original
    bank_2.detach().numpy()
]

print("Cosine similarities between the 'bank' words:")
cosine_similarities(banks)
print("The cosine similarity between the BERT embeddings for 'bank' in these two sentences is 0.74.")

### END YOUR CODE

Cosine similarities between the 'bank' words:
	1.0	0.74
	0.74	1.0
The cosine similarity between the BERT embeddings for 'bank' in these two sentences is 0.74.


**QUESTION:**

1.h How does this relate to the cosine similarity of 'this' (in sentence 1) and the first 'the' (in sentence 2). Compute their cosine similarity.


In [17]:
### YOUR CODE HERE

#h.  -> get the vectors and calculate cosine similarity

# Getting the cosine similarities between the BERT embeddings for "this" and "the" in the first and second sentences, respectively
this_1 = bert_output[0][0, 1]
the_2 = bert_output[0][1, 1]

this_the = [
    this_1.detach().numpy(),
    the_2.detach().numpy()
]

print("Cosine similarities between the 'this' and 'the':")
cosine_similarities(this_the)
print("The cosine similarity between the BERT embeddings for 'this' in the first sentence and 'the' in the second sentence is 0.81.")
print("Therefore, it appears that these words are more closely related semantically than 'bank' is between the two sentences.")

### END YOUR CODE

Cosine similarities between the 'this' and 'the':
	1.0	0.81
	0.81	1.0
The cosine similarity between the BERT embeddings for 'this' in the first sentence and 'the' in the second sentence is 0.81.
Therefore, it appears that these words are more closely related semantically than 'bank' is between the two sentences.


### 2. Testing Different Pre-Trained BERT Models

In the live session we discussed classification with the `bert-base-cased` model, using the Huggingface class BertForSequenceClassification, which comes with a new output layer for our task that we need to train on our dataset.

We're going to try different pre-trained models now. Like in the lesson 4 notebook, we'll want to fine-tune each model on our IMDB reviews dataset and compare them with a metric like the validation accuracy. We'll use the model class AutoModelForSequenceClassification, which is equivalent to BertForSequenceClassification, but works for other similar architectures too.

Let's write the code we'll need as a function that takes the model and tokenizer as arguments, along with the raw train and dev data. The function will need to tokenize the inputs using the provided tokenizer, so that we can repeat the same code for different pre-trained models. Then the function should create the training args and trainer class, and call trainer.train().

The other hyperparameters you'll need are provided in the function definition, including batch_size and num_epochs. You should use the default values provided for those. Use the function provided below for compute_metrics.

For now, keep all layers of the pre-trained models you load unfrozen.

In [18]:
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [19]:
# Getting the current directory
import os
os.getcwd()

'/content'

In [21]:
# Mounting my Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
# Changing to base directory and saving the outputs there
saved_output_dir = '/content/drive/MyDrive/Colab Notebooks/w266/w266 Assignments/w266 Assignment 2'
os.chdir(saved_output_dir)
os.getcwd()

'/content/drive/MyDrive/Colab Notebooks/w266/w266 Assignments/w266 Assignment 2'

In [23]:
def fine_tune_classification_model(classification_model,
                                   tokenizer,
                                   train_data,
                                   dev_data,
                                   batch_size = 16,
                                   num_epochs = 2):
    """
    Preprocess the data using the given tokenizer (we've give you the code for that part).
    Create the training arguments and trainer for the given model and data (write your code for that).
    Then train it.
    """

    preprocessed_train_data = train_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})
    preprocessed_dev_data = dev_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})

    ### YOUR CODE HERE

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to='none'
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics
    )



    ### END YOUR CODE

    trainer.train()

Let's try BERT-base-case first, the same model that was used in the lesson 4 notebook.

In [ ]:
"""
Show the output from training BERT-base-cased on the IMDB movie reviews dataset.
"""

model_checkpoint_name = "bert-base-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

fine_tune_classification_model(bert_classification_model, bert_tokenizer, imdb_train_dataset, imdb_dev_dataset)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.360500,0.331484,0.855600
2,0.207800,0.391858,0.860200


Often, one of the first choices you have is what pre-trained model you'll want to use. There are quite a few options, especially because other researchers and practitioners fine-tune their own versions of existing models and sometimes make theirs available for others to continue building on.

You can search through models available on [Huggingface at this website](https://huggingface.co/models?pipeline_tag=text-classification&sort=trending). Some models were made by Huggingface or other large companies/organizations; other models may have been uploaded by individual users. Notice the search tags on the left, we've already clicked the tag for "Text Classification" in the link above. You should see various versions of BERT-style models.

For our IMDB classification, we might want to try a model that has been trained on another dataset related to sentiment or emotions. We also want to find models that have a complete model card with documentation about the model architecture and how it was trained, and potentially a link to an associated research paper, and/or a good number of downloads and likes.

Take a look at this model: [cardiffnlp/twitter-roberta-base-sentiment](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment). It's a RoBERTa model (similar to BERT with slightly different pre-training, often popular for classification tasks), that has already been fine-tuned on the TweetEval benchmark set of tasks for sentiment analysis.

The model card indicates that there is an updated version of this model now available. Follow the link to the latest version of the model, and look at that most recent model's card to answer the following questions. Then load that most recent model to train on our task.

**QUESTION:**

 2.a What is the model checkpoint name for the most recent version of this Twitter Roberta-base sentiment analysis model? (Copy and paste the model checkpoint name into the answers file. It should be the full name that you put inside the quotes to load the file below.)
 - "cardiffnlp/twitter-roberta-base-sentiment-latest"


 **QUESTION:**

 2.b Approximately how many tweets was this latest model trained on? (Put the answer in the answers file. You can use the abbreviation for millions like in the model card, e.g. a number like 12M or 85M.)
- 124M


 **QUESTION:**

 2.c What is the title of the published reference paper for this most recent model? (Copy the full title of the paper and paste it into the answers file.)
 - "TimeLMs: Diachronic Language Models from Twitter"

In [ ]:
"""
Show the output from training the most recent Twitter RoBERTa sentiment model on the IMDB movie reviews dataset.
Insert the model checkpoint name for the latest version of that model below.
"""

### YOUR CODE HERE

model_checkpoint_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"


### END YOUR CODE


bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

fine_tune_classification_model(bert_classification_model, bert_tokenizer, imdb_train_dataset, imdb_dev_dataset)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.319300,0.266863,0.891000
2,0.188900,0.329761,0.897200


**QUESTION:**

2.d What is the final validation accuracy that you observed for the Twitter RoBERTa sentiment-trained model after training for 2 epochs? (Copy and paste the decimal value for the final validation accuracy, e.g. a number like 0.567 or 0.876. Use up to 5 significant digits, though fewer is fine if the output shown in the notebook only has 3 or 4. Put the answer in the answers file; it should match the value shown in your output in this notebook.)
- 0.89720


**QUESTION:**

2.e Did the Twitter RoBERTa sentiment-trained model do better or worse or the same as the BERT-base?
- Better


**(Answer 2.f below but do NOT enter your sentences in the answers file)**

**QUESTION:**

2.f Why do you think that happened? (Put your two to three sentence answer in the cell below.)

Please answer 2.f in two to three sentences right here:

** BEGIN Q 2.f ANSWER HERE **

One reason this could have happened is because the Twitter RoBERTa model was already fine-tuned on sentiment-related tasks (TweetEval) before we applied it here to the IMDB dataset, meaning it already learned sentiment-specific patterns in the text. Even though tweets and movie reviews are objectively different, perhaps the emotional tones and undertones learned from tweets is still transferable to our use case. Conversely, the BERT-base model is more of a general model without the sentiment-specific fine-tuning, meaning it had to learn these sentiment patterns from scratch from the IMDB dataset.


** END Q 2.f ANSWER HERE. **


### 3. Unfreezing Different Pre-Trained Layers

In the lesson 4 notebook, we tested freezing most or all of the pre-trained BERT model layers. We used the .named_parameters() method, looking at the specific names of each set of model parameters.

As in the lesson notebook, we will always want to make sure we keep the classification layer parameters unfrozen, since those need to be trained for our specific task. We will also keep the pooler layer unfrozen, since it's next closest to the classification layer and was only pre-trained in standard BERT models with the next sentence prediction task.

For the remaining layers, what happens if we unfreeze lower transformer blocks and keep higher transformer blocks frozen (the opposite of what we did in the lesson notebook)? What if we instead try unfreezing specific types of layers within each transformer block, e.g. all of the self attention layers, or all of the dense layers?

Let's modify our fine-tuning function, to add an argument for the layers that we want to train. We'll make that argument a list of strings, and we'll set the default to just unfreeze the classification layer. You'll need to write the code to compare those strings to the names of the model parameters (after loading the specified model) and freeze all parameters that don't match (as in the lesson 4 notebook).

In [ ]:
# Refresh your memory on what the parameter names look like
for name, param in bert_classification_model.named_parameters():
    print(name)

roberta.embeddings.word_embeddings.weight
roberta.embeddings.position_embeddings.weight
roberta.embeddings.token_type_embeddings.weight
roberta.embeddings.LayerNorm.weight
roberta.embeddings.LayerNorm.bias
roberta.encoder.layer.0.attention.self.query.weight
roberta.encoder.layer.0.attention.self.query.bias
roberta.encoder.layer.0.attention.self.key.weight
roberta.encoder.layer.0.attention.self.key.bias
roberta.encoder.layer.0.attention.self.value.weight
roberta.encoder.layer.0.attention.self.value.bias
roberta.encoder.layer.0.attention.output.dense.weight
roberta.encoder.layer.0.attention.output.dense.bias
roberta.encoder.layer.0.attention.output.LayerNorm.weight
roberta.encoder.layer.0.attention.output.LayerNorm.bias
roberta.encoder.layer.0.intermediate.dense.weight
roberta.encoder.layer.0.intermediate.dense.bias
roberta.encoder.layer.0.output.dense.weight
roberta.encoder.layer.0.output.dense.bias
roberta.encoder.layer.0.output.LayerNorm.weight
roberta.encoder.layer.0.output.LayerNorm

In [24]:
def fine_tune_classif_model_freeze_layers(classification_model,
                                          tokenizer,
                                          train_data,
                                          dev_data,
                                          layers_to_train = ["classifier."],
                                          max_sequence_length=MAX_SEQUENCE_LENGTH,
                                          batch_size = 16,
                                          num_epochs = 2):
    """
    Freeze any parameters inside the given model that have a name containing one of the
    strings in the "layers_to_freeze" list.
    Then specify the training arguments and trainer for the given model and data.
    Then train it.
    """

    preprocessed_train_data = train_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})
    preprocessed_dev_data = dev_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})

    ### YOUR CODE HERE

    # Only allowing the training of layers in layers_to_train
    for name, param in classification_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to='none'
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics
    )

    ### END YOUR CODE

    trainer.train()

We'll go back to using bert-base-cased for this part. First, try freezing the parameters in transformer layers 1-11 (including all parameters with "layer.#" in the name). That means you're leaving unfrozen the initial embedding layers, the first transformer layer (numbered 0), and the classification layer.

Unfreezing the bottom transformer layer(s) rather than the top one(s) is uncommon, but it's always good to try to understand why. Since we're learning, we'll try doing it this way and see what happens. We've given you the code for this exercise, so that the way to specify layers_to_freeze is clear.

In [ ]:
"""
Show the output from training a BERT-base-cased classification model, when unfreezing
only the parameters in the embedding layers, first transformer layer (layer 0), and classifier layer.
"""

model_checkpoint_name = "bert-base-cased"

bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

layers_to_train = ["embeddings.", "layer.0.", "classifier."]

fine_tune_classif_model_freeze_layers(
    bert_classification_model,
    bert_tokenizer,
    imdb_train_dataset,
    imdb_dev_dataset,
    layers_to_train
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.449700,0.440541,0.800600
2,0.351900,0.434646,0.810400


 **QUESTION:**

3.a What is the final validation accuracy that you observed for this lowest level unfrozen version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- 0.81040




Now try two more versions, this time choosing which layers to train yourself. Instead of focusing on the number of the transformer block (layer.#), focus on the type of layer within each block (the stuff that comes after layer.# in the name).

Keep the pooler and classification layers unfrozen in all model versions. Your options to also train include the initial embedding layers and the different components within the transformer blocks (e.g. self attention matrices, dense layers, layer norms).

Try to find one combination that does better than the version you just ran above (higher validation accuracy after 2 epochs), without much more overfitting (training_loss / eval_loss > 0.7). Also try to find one version that overfits a lot more after 2 epochs (training_loss / eval_loss < 0.5).

In [ ]:
for name, param in bert_classification_model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [ ]:
"""
Show the output from training a particular model on the IMDB movie reviews dataset.
Choose layers to train that lead the model to perform better than the one in question 3.a, without overfitting much more.
"""

model_checkpoint_name = "bert-base-cased"

bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

### YOUR CODE HERE

layers_to_train = ["attention.self.query", "attention.self.value", "output.LayerNorm", "bert.pooler", "classifier"]    #ANY STRINGS THAT MATCH SOME LAYERS ARE OK


### END YOUR CODE


fine_tune_classif_model_freeze_layers(
    bert_classification_model,
    bert_tokenizer,
    imdb_train_dataset,
    imdb_dev_dataset,
    layers_to_train
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.343400,0.318504,0.861600
2,0.269000,0.326507,0.865800


 **QUESTION:**

3.b What is the final training loss that you observed for this better performing version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- 0.26900


3.c What is the final validation loss that you observed for this better performing version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- 0.32650


3.d What is the ratio of your final training loss/final validation loss? For this better version the ratio must be greater than 0.7.
- 0.82389


3.e What is the final validation accuracy that you observed for this better performing version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- 0.86580

In [25]:
"""
Show the output from training a particular model on the IMDB movie reviews dataset.
Choose layers to train that lead the model to overfit.
"""

model_checkpoint_name = "bert-base-cased"

bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_name)
bert_classification_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint_name)

### YOUR CODE HERE

layers_to_train = ["bert.embeddings.", "intermediate.dense", "attention.self", "LayerNorm", "bert.pooler", "classifier"]


### END YOUR CODE


fine_tune_classif_model_freeze_layers(
    bert_classification_model,
    bert_tokenizer,
    imdb_train_dataset,
    imdb_dev_dataset,
    layers_to_train
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

 **QUESTION:**

3.f What is the final training loss that you observed for this overfitting version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- df


3.g What is the final validation loss that you observed for this overfitting version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- df


3.h What is the ratio of your final training loss/final validation loss? For this overfitting version the ratio must be less than 0.5.
- df


3.i What is the final validation accuracy that you observed for this overfitting version of the BERT classification model after training for 2 epochs? (Copy and paste the decimal value into the answers file, as instructed in 2.b)
- df

## Congratulations... You are done!